# NB00 — Data Preprocessing

Prepares dataset metadata, Tufts gaze maps (48×48 `.npy`), and train/val/test splits for the dual-arm gaze-regularized LoRA pipeline.

**Outputs (plan §8):**
- `splits/train_split.csv` — Tufts train (800) + OralXray train+val (~12,688)
- `splits/val_split.csv` — Tufts val (100)
- `splits/test_split.csv` — Tufts test (100)
- `splits/dentex_test_split.csv` — all DENTEX images (held out)
- `splits/sweep_split.csv` — 10% stratified sweep subset (NB02)
- `splits/oralxray_test_split.csv` — OralXrays held-out test slice (NB07 §7.1 forgetting check)
- `gaze_maps_48x48/*.npy` — one per Tufts image with expert gaze

**Prerequisite:** run `novelty_checks.ipynb` first (`week1_decision_log.json` should exist).


In [ ]:
import os, sys, json

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT = '/content/drive/MyDrive/DENTAL'
except ImportError:
    # Running locally on Windows or Linux
    cwd = os.getcwd()
    PROJECT = os.path.abspath(os.path.join(cwd, '..')) if os.path.basename(cwd) == 'Notebooks' else cwd

NOTEBOOKS_DIR = os.path.join(PROJECT, 'Notebooks')
sys.path.insert(0, PROJECT)
sys.path.insert(0, NOTEBOOKS_DIR)
os.chdir(PROJECT)

with open('config.json', 'r') as f:
    CONFIG = json.load(f)

print('Working directory:', os.getcwd())
print('Resolution:', CONFIG['chosen_resolution'])
print('Gaze strategy:', CONFIG.get('gaze_entropy_decision', 'DUAL_ARM'))


Mounted at /content/drive
Working directory: /content/drive/MyDrive/DENTAL
Resolution: 224
Gaze strategy: DUAL_ARM


In [ ]:
# === Unpack dataset from Data zip (Colab) ===
# Expects zip(s) in PROJECT/Data zip/ with the same layout as PROJECT/data/
import glob
import os
import zipfile

DATA_ZIP_DIR = os.path.join(PROJECT, 'Data zip')
marker_lower = os.path.join(PROJECT, 'data', 'tdd', 'expert', 'expert.json')
marker_upper = os.path.join(PROJECT, 'data', 'tdd', 'Expert', 'expert.json')
DATA_MARKER = marker_upper if os.path.exists(marker_upper) else marker_lower

if os.path.exists(DATA_MARKER):
    print(f'Dataset already present: {os.path.join(PROJECT, "data")}')
else:
    if not os.path.isdir(DATA_ZIP_DIR):
        raise FileNotFoundError(f'Missing folder: {DATA_ZIP_DIR}')
    zip_paths = sorted(glob.glob(os.path.join(DATA_ZIP_DIR, '*.zip')))
    if not zip_paths:
        raise FileNotFoundError(f'No .zip files found in {DATA_ZIP_DIR}')
    for zip_path in zip_paths:
        print(f'Unzipping {os.path.basename(zip_path)} -> {PROJECT} ...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(PROJECT)
    if not os.path.exists(DATA_MARKER):
        raise FileNotFoundError(
            'Unzip finished but data/tdd/Expert/expert.json not found. '
            'Ensure the archive contains a top-level data/ folder.'
        )
    print(f'Dataset ready under {os.path.join(PROJECT, "data")}')


In [ ]:
!pip install -q pillow opencv-python pandas numpy scikit-learn tqdm pyyaml

In [ ]:
import glob
import os
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

RANDOM_SEED = int(CONFIG.get('random_seed', 42))
GAZE_SIZE = 48

PATHS = {
    'tufts_expert_json': 'data/tdd/Expert/expert.json',
    'tufts_radiographs': 'data/tdd/Radiographs',
    'tufts_gaze_gray': 'data/tdd/Expert/gaze_map/gray',
    'oralxray_root': 'data/oralxrays-91',
    'oralxray_train_json': 'data/oralxrays-91/annotations/instances_train2017.json',
    'oralxray_val_json': 'data/oralxrays-91/annotations/instances_val2017.json',
    'dentex_root': 'data/dentex',
    'dentex_val_json': 'data/dentex/validation_triple.json',
    'dentex_train_disease_json': 'data/dentex/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json',
    'dentex_test_label_dir': 'data/dentex/test_data/disease/label',
}

for k, v in PATHS.items():
    ok = os.path.exists(v)
    print(f"{k:28s} {'OK' if ok else 'MISSING':7s}  {v}")
    if not ok and k not in ('dentex_test_label_dir',):
        raise FileNotFoundError(f"Missing required path: {v}")

os.makedirs('splits', exist_ok=True)
os.makedirs('gaze_maps_48x48', exist_ok=True)


tufts_expert_json            OK       data/tdd/Expert/expert.json
tufts_radiographs            OK       data/tdd/Radiographs
tufts_gaze_gray              OK       data/tdd/Expert/gaze_map/gray
oralxray_root                OK       data/oralxrays-91
oralxray_train_json          OK       data/oralxrays-91/annotations/instances_train2017.json
oralxray_val_json            OK       data/oralxrays-91/annotations/instances_val2017.json
dentex_root                  OK       data/dentex
dentex_val_json              OK       data/dentex/validation_triple.json
dentex_train_disease_json    OK       data/dentex/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json
dentex_test_label_dir        OK       data/dentex/test_data/disease/label


## 1 — Parse Tufts TDD (`expert.json`)

Binary label rule (validated in Week 1): `has_lesion_gt = 1` when any `Label.objects[].title` is not `"None"`.


In [ ]:
def parse_tufts_expert_json(json_path):
    with open(json_path, 'r') as f:
        records_raw = json.load(f)

    rows = []
    for rec in records_raw:
        ext_id = rec.get('External ID', '')
        image_id = os.path.splitext(ext_id)[0]
        objs = rec.get('Label', {}).get('objects', [])
        titles = [str(o.get('title', '')).strip().lower() for o in objs if isinstance(o, dict)]
        has_lesion = int(any(t and t != 'none' for t in titles))

        image_name = ext_id if ext_id else f"{image_id}.JPG"
        image_path = os.path.join(PATHS['tufts_radiographs'], image_name)
        if not os.path.exists(image_path):
            for ext in ('.JPG', '.jpg', '.png'):
                cand = os.path.join(PATHS['tufts_radiographs'], f"{image_id}{ext}")
                if os.path.exists(cand):
                    image_path = cand
                    break

        rows.append({
            'id': image_id,
            'image_path': image_path.replace('\\', '/'),
            'dataset': 'tufts',
            'has_lesion_gt': has_lesion,
            'has_gaze_map': 0,
            'gaze_map_path': '',
        })

    df = pd.DataFrame(rows)
    assert 'has_lesion_gt' in df.columns
    return df

tufts_meta = parse_tufts_expert_json(PATHS['tufts_expert_json'])
print(f"Tufts records: {len(tufts_meta)}  (lesion={tufts_meta['has_lesion_gt'].sum()}, healthy={(tufts_meta['has_lesion_gt']==0).sum()})")
missing_img = (~tufts_meta['image_path'].map(os.path.exists)).sum()
print(f"Missing radiograph files: {missing_img}")
assert len(tufts_meta) == 1000, f"Expected 1000 Tufts images, got {len(tufts_meta)}"


Tufts records: 1000  (lesion=340, healthy=660)
Missing radiograph files: 0


## 2 — Expert gaze JPEGs → 48×48 `.npy`

Resize grayscale gaze heatmaps to 48×48, normalize to `[0, 1]`, save under `gaze_maps_48x48/`.


In [ ]:
def load_gaze_jpeg(gaze_path, size=GAZE_SIZE):
    gaze_img = Image.open(gaze_path).convert('L')
    gaze_array = np.array(gaze_img, dtype=np.float32)
    if gaze_array.max() > 1.0:
        gaze_array = gaze_array / 255.0
    gaze_48 = cv2.resize(gaze_array, (size, size), interpolation=cv2.INTER_LINEAR)
    return gaze_48.astype(np.float32)

gaze_dir = PATHS['tufts_gaze_gray']
gaze_saved, gaze_missing = 0, []

for _, row in tqdm(tufts_meta.iterrows(), total=len(tufts_meta), desc='gaze maps'):
    image_id = str(row['id'])
    gaze_path = None
    for ext in ('.JPG', '.jpg', '.png'):
        cand = os.path.join(gaze_dir, f"{image_id}{ext}")
        if os.path.exists(cand):
            gaze_path = cand
            break
    if gaze_path is None:
        gaze_missing.append(image_id)
        continue

    gaze = load_gaze_jpeg(gaze_path)
    out_path = f"gaze_maps_48x48/{image_id}.npy"
    np.save(out_path, gaze)
    tufts_meta.loc[tufts_meta['id'] == image_id, 'has_gaze_map'] = 1
    tufts_meta.loc[tufts_meta['id'] == image_id, 'gaze_map_path'] = out_path
    gaze_saved += 1

print(f"Saved {gaze_saved} gaze maps -> gaze_maps_48x48/")
print(f"Missing gaze JPEGs: {len(gaze_missing)}")
if gaze_missing:
    pd.DataFrame({'id': gaze_missing}).to_csv('splits/nb00_missing_gaze.csv', index=False)


gaze maps:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved 1000 gaze maps -> gaze_maps_48x48/
Missing gaze JPEGs: 0


## 3 — Parse OralXrays-91 (COCO)

`has_lesion_gt = 1` when the image has ≥1 COCO annotation (any finding). Train + val COCO splits are merged into the **training pool** only.


In [ ]:
def find_oralxray_image(filename):
    for sub in ('train2017 (1)', 'val2017 (1)', 'train2017', 'val2017'):
        cand = os.path.join(PATHS['oralxray_root'], sub, filename)
        if os.path.exists(cand):
            return cand.replace('\\', '/')
    matches = glob.glob(os.path.join(PATHS['oralxray_root'], '**', filename), recursive=True)
    return matches[0].replace('\\', '/') if matches else ''

def parse_coco_annotations(json_path, split_tag):
    with open(json_path, 'r') as f:
        coco = json.load(f)

    img_to_anns = defaultdict(list)
    for ann in coco.get('annotations', []):
        img_to_anns[ann['image_id']].append(ann)

    rows = []
    for img in coco.get('images', []):
        anns = img_to_anns.get(img['id'], [])
        rows.append({
            'id': str(img['id']),
            'image_path': find_oralxray_image(img['file_name']),
            'dataset': 'oralxrays',
            'has_lesion_gt': int(len(anns) > 0),
            'has_gaze_map': 0,
            'gaze_map_path': '',
            'coco_split': split_tag,
        })
    return pd.DataFrame(rows)

oralxray_train = parse_coco_annotations(PATHS['oralxray_train_json'], 'train')
oralxray_val = parse_coco_annotations(PATHS['oralxray_val_json'], 'val')
oralxray_all = pd.concat([oralxray_train, oralxray_val], ignore_index=True)

missing_ox = (oralxray_all['image_path'] == '').sum()
print(f"OralXray images: {len(oralxray_all)} (train={len(oralxray_train)}, val={len(oralxray_val)})")
print(f"  lesion={oralxray_all['has_lesion_gt'].sum()}, healthy={(oralxray_all['has_lesion_gt']==0).sum()}")
print(f"  missing image files: {missing_ox}")
assert missing_ox == 0, 'Some OralXray images were not found on disk'


OralXray images: 12688 (train=10000, val=2688)
  lesion=12449, healthy=239
  missing image files: 0


## 4 — Parse DENTEX (held out)

Image-level `has_lesion_gt = 1` when any annotation has `category_id_3 > 0` (Caries / Periapical / Deep Caries). Test-set LabelMe JSONs use disease/lezyon prefixes. Unlabelled training panoramas are kept with `has_lesion_gt = -1`.


In [ ]:
def dentex_has_lesion_from_anns(anns):
    return int(any(a.get('category_id_3', 0) > 0 for a in anns))

def parse_dentex_coco(json_path, image_root):
    with open(json_path, 'r') as f:
        coco = json.load(f)

    img_to_anns = defaultdict(list)
    for ann in coco.get('annotations', []):
        img_to_anns[ann['image_id']].append(ann)

    rows = []
    for img in coco.get('images', []):
        fname = img['file_name']
        matches = glob.glob(os.path.join(image_root, '**', fname), recursive=True)
        image_path = matches[0].replace('\\', '/') if matches else ''
        anns = img_to_anns.get(img['id'], [])
        rows.append({
            'id': os.path.splitext(fname)[0],
            'image_path': image_path,
            'dataset': 'dentex',
            'has_lesion_gt': dentex_has_lesion_from_anns(anns),
            'has_gaze_map': 0,
            'gaze_map_path': '',
        })
    return pd.DataFrame(rows)

def parse_dentex_test_disease(label_dir):
    rows = []
    for lp in sorted(glob.glob(os.path.join(label_dir, '*.json'))):
        with open(lp, 'r') as f:
            data = json.load(f)
        fname = data.get('imagePath') or (os.path.basename(lp).replace('.json', '.png'))
        matches = glob.glob(os.path.join(PATHS['dentex_root'], '**', fname), recursive=True)
        image_path = matches[0].replace('\\', '/') if matches else ''
        shapes = data.get('shapes', [])
        has_lesion = 0
        for s in shapes:
            lab = str(s.get('label', '')).lower()
            if lab.startswith('7-lezyon') or lab.startswith('1-') or 'caries' in lab or 'lezyon' in lab:
                has_lesion = 1
                break
        rows.append({
            'id': os.path.splitext(fname)[0],
            'image_path': image_path,
            'dataset': 'dentex',
            'has_lesion_gt': has_lesion,
            'has_gaze_map': 0,
            'gaze_map_path': '',
        })
    return pd.DataFrame(rows)

dentex_val = parse_dentex_coco(PATHS['dentex_val_json'], PATHS['dentex_root'])
dentex_train_dis = parse_dentex_coco(PATHS['dentex_train_disease_json'], PATHS['dentex_root'])
dentex_test = parse_dentex_test_disease(PATHS['dentex_test_label_dir'])

labelled = pd.concat([dentex_val, dentex_train_dis, dentex_test], ignore_index=True)
labelled = labelled.drop_duplicates(subset=['id'], keep='first')
labelled_ids = set(labelled['id'])

all_pngs = glob.glob(os.path.join(PATHS['dentex_root'], '**', '*.png'), recursive=True)
unlabelled_rows = []
for p in all_pngs:
    image_id = os.path.splitext(os.path.basename(p))[0]
    if image_id in labelled_ids:
        continue
    unlabelled_rows.append({
        'id': image_id,
        'image_path': p.replace('\\', '/'),
        'dataset': 'dentex',
        'has_lesion_gt': -1,
        'has_gaze_map': 0,
        'gaze_map_path': '',
    })

dentex_meta = pd.concat([labelled, pd.DataFrame(unlabelled_rows)], ignore_index=True)
dentex_meta = dentex_meta.drop_duplicates(subset=['image_path'], keep='first')

known = dentex_meta['has_lesion_gt'] >= 0
print(f"DENTEX total images: {len(dentex_meta)}")
print(f"  labelled: {known.sum()} (lesion={(dentex_meta.loc[known,'has_lesion_gt']==1).sum()}, healthy={(dentex_meta.loc[known,'has_lesion_gt']==0).sum()})")
print(f"  unlabelled (has_lesion_gt=-1): {(~known).sum()}")


DENTEX total images: 1871
  labelled: 1005 (lesion=936, healthy=69)
  unlabelled (has_lesion_gt=-1): 866


## 5 — Tufts stratified splits (800 / 100 / 100)

DENTEX is **never** included in train/val. OralXray val COCO images are folded into the training pool.


In [ ]:
assert len(tufts_meta) == 1000

tufts_remain, tufts_test = train_test_split(
    tufts_meta,
    test_size=100,
    stratify=tufts_meta['has_lesion_gt'],
    random_state=RANDOM_SEED,
)
tufts_train, tufts_val = train_test_split(
    tufts_remain,
    test_size=100,
    stratify=tufts_remain['has_lesion_gt'],
    random_state=RANDOM_SEED,
)

print('Tufts split sizes:')
print(f"  train: {len(tufts_train)}  (lesion={tufts_train['has_lesion_gt'].sum()})")
print(f"  val:   {len(tufts_val)}  (lesion={tufts_val['has_lesion_gt'].sum()})")
print(f"  test:  {len(tufts_test)}  (lesion={tufts_test['has_lesion_gt'].sum()})")
assert len(tufts_train) == 800 and len(tufts_val) == 100 and len(tufts_test) == 100

# v23 Upgrade: Clean separation of Phase 1 (OralXrays) and Phase 2 (Tufts TDD)
# train_pool is strictly Tufts Train (800 images) for Phase 2 fine-tuning.
train_pool = tufts_train.copy()

# Stratified 80% train / 10% val / 10% test split for OralXrays (matching Tufts 80/10/10 structure)
oral_remain, oral_test = train_test_split(
    oralxray_all, test_size=0.10, stratify=oralxray_all['has_lesion_gt'], random_state=RANDOM_SEED
)
oral_train, oral_val = train_test_split(
    oral_remain, test_size=1/9, stratify=oral_remain['has_lesion_gt'], random_state=RANDOM_SEED
)
print(f"OralXrays split sizes:\n  train (80% Phase 1): {len(oral_train)}\n  val (10%): {len(oral_val)}\n  test (10% holdout): {len(oral_test)}")

for df, name in [
    (train_pool, 'tufts_train'),
    (tufts_val, 'tufts_val'),
    (tufts_test, 'tufts_test'),
    (oral_train, 'oral_train'),
    (oral_val, 'oral_val'),
    (oral_test, 'oral_test'),
    (dentex_meta, 'dentex'),
]:
    assert 'has_lesion_gt' in df.columns, f"{name} missing has_lesion_gt"
    print(f"OK  {name:12s}  n={len(df):5d}  has_lesion_gt column present")


Tufts split sizes:
  train: 800  (lesion=272)
  val:   100  (lesion=34)
  test:  100  (lesion=34)
OK  train   n=13488  has_lesion_gt column present
OK  val     n=  100  has_lesion_gt column present
OK  test    n=  100  has_lesion_gt column present
OK  dentex  n= 1871  has_lesion_gt column present


## 6 — Optional dedup guard (Week 1)

Cross-dataset near-duplicates must not appear in training. Week 1 reported 0 cross-dataset pairs.


In [ ]:
dedup_path = 'dedup_report.csv'
if os.path.exists(dedup_path):
    dupes = pd.read_csv(dedup_path)
    cross = dupes[dupes['dataset_a'] != dupes['dataset_b']] if len(dupes) else dupes
    print(f"Near-duplicate pairs: {len(dupes)} | cross-dataset: {len(cross)}")
    if len(cross):
        raise RuntimeError('Cross-dataset duplicates found — resolve before training (see novelty_checks.ipynb)')
else:
    print('dedup_report.csv not found — skipping (run novelty_checks.ipynb to generate)')


dedup_report.csv not found — skipping (run novelty_checks.ipynb to generate)


## 7 — Save outputs


In [11]:
from sklearn.model_selection import StratifiedKFold

# Assign 5-fold cross-validation fold IDs strictly within Tufts Train (800 images)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
train_pool['fold'] = -1
for fold_idx, (_, val_idx) in enumerate(skf.split(train_pool, train_pool['has_lesion_gt'])):
    train_pool.loc[train_pool.index[val_idx], 'fold'] = fold_idx

print(f"Tufts Train Fold distribution:\n{train_pool['fold'].value_counts().sort_index()}")
assert (train_pool['fold'] >= 0).all(), "Some rows in tufts_train were not assigned a fold"

# For OralXrays Phase 1 train, assign fold=0 for schema compatibility
oral_train['fold'] = 0
oral_val['fold'] = -1
oral_test['fold'] = -1

cols_with_fold = ['id', 'image_path', 'dataset', 'has_lesion_gt', 'has_gaze_map', 'gaze_map_path', 'fold']
cols_no_fold = ['id', 'image_path', 'dataset', 'has_lesion_gt', 'has_gaze_map', 'gaze_map_path']

train_save = train_pool[cols_with_fold]
val_save = tufts_val[cols_no_fold]
test_save = tufts_test[cols_no_fold]
dentex_save = dentex_meta[cols_no_fold]

os.makedirs('splits', exist_ok=True)
train_save.to_csv('splits/train_split.csv', index=False)
val_save.to_csv('splits/val_split.csv', index=False)
test_save.to_csv('splits/test_split.csv', index=False)
dentex_save.to_csv('splits/dentex_test_split.csv', index=False)

# v23 Dedicated OralXrays Phase 1 Pre-training split (80/10/10)
oral_train[cols_with_fold].to_csv('splits/oralxray_train_split.csv', index=False)
oral_val[cols_no_fold].to_csv('splits/oralxray_val_split.csv', index=False)
oral_test[cols_with_fold].to_csv('splits/oralxray_test_split.csv', index=False)

# v23 Sweep split: 20% stratified subset of Tufts train (160 images) for 10x faster Optuna execution
_, sweep_df = train_test_split(
    train_pool, test_size=0.20, stratify=train_pool['has_lesion_gt'], random_state=RANDOM_SEED)
sweep_df[cols_with_fold].to_csv('splits/sweep_split.csv', index=False)

print(f"sweep_split (Tufts only): {len(sweep_df)}  |  oralxray_train_split: {len(oral_train)}  |  oralxray_val_split: {len(oral_val)}  |  oralxray_test_split: {len(oral_test)}")

summary = {
    'random_seed': RANDOM_SEED,
    'tufts_train': int(len(tufts_train)),
    'tufts_val': int(len(tufts_val)),
    'tufts_test': int(len(tufts_test)),
    'oralxray_train_phase1': int(len(oral_train)),
    'oralxray_val_phase1': int(len(oral_val)),
    'oralxray_test_holdout': int(len(oral_test)),
    'oralxray_total': int(len(oralxray_all)),
    'dentex_total': int(len(dentex_meta)),
    'dentex_labelled': int((dentex_meta['has_lesion_gt'] >= 0).sum()),
    'gaze_maps_saved': int(gaze_saved),
    'gaze_maps_missing': int(len(gaze_missing)),
    'sweep_subset_tufts': int(len(sweep_df)),
}
with open('splits/nb00_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved v23 canonical splits:')
for p in [
    'splits/train_split.csv',
    'splits/val_split.csv',
    'splits/test_split.csv',
    'splits/dentex_test_split.csv',
    'splits/oralxray_train_split.csv',
    'splits/oralxray_val_split.csv',
    'splits/oralxray_test_split.csv',
    'splits/sweep_split.csv',
    'splits/nb00_summary.json',
]:
    print(f"  {p}")
print(json.dumps(summary, indent=2))
print('\nNB00 complete  proceed to NB01_model.ipynb')


Fold distribution:
fold
0    2698
1    2698
2    2698
3    2697
4    2697
Name: count, dtype: int64
sweep_split: 1349  |  oralxray_test_split: 635
Saved:
  splits/train_split.csv
  splits/val_split.csv
  splits/test_split.csv
  splits/dentex_test_split.csv
  splits/sweep_split.csv
  splits/oralxray_test_split.csv
  splits/nb00_summary.json
{
  "random_seed": 42,
  "tufts_train": 800,
  "tufts_val": 100,
  "tufts_test": 100,
  "oralxray_in_train_pool": 12688,
  "train_pool_total": 13488,
  "dentex_total": 1871,
  "dentex_labelled": 1005,
  "gaze_maps_saved": 1000,
  "gaze_maps_missing": 0,
  "sweep_subset": 1349,
  "oralxray_test": 635
}

NB00 complete — proceed to NB01_model.ipynb
